# Qubitization walks

Adapted from Qualtran v0.7.0 `qubitization/qubitization_walk_operator.ipynb`
and its two integral fixtures. Copyright 2023 and 2024 Google LLC, Apache-2.0.
Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.

For H = sum_l w_l U_l, PREPARE encodes sqrt(w_l/lambda) on a selection word,
and SELECT applies U_l to the target. The walk W = (2|L><L| - I) SELECT
encodes E/lambda as cos(arg(eigenvalue(W))). Its prepared signal and any live
logical junk retain explicit ownership across successive applications.


In [ ]:
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from typing import cast

import numpy as np
import torch
from IPython import display as notebook_display

from flagquantum import Circuit
from flagquantum.algorithms.primitives.identity_prepare import IdentityPrepare
from flagquantum.algorithms.primitives.ising_walk import ising_walk
from flagquantum.algorithms.primitives.multiplexers.select_pauli_lcu import (
    SelectPauliLCU,
)
from flagquantum.algorithms.primitives.multiplexers.thc_resource_model import (
    select_thc_signature,
)
from flagquantum.algorithms.primitives.prepare_oracle import PrepareOracle
from flagquantum.algorithms.primitives.qubitization_walk import (
    QubitizationWalk,
    source_walk_signature,
)
from flagquantum.algorithms.primitives.select_block_encoding import SelectBlockEncoding
from flagquantum.algorithms.primitives.sparse_chemistry_walk import SparseChemistryWalk
from flagquantum.algorithms.primitives.symbolic_alias_preparation import (
    AliasPreparationShape,
)
from flagquantum.algorithms.primitives.thc_chemistry_data import thc_chemistry_data
from flagquantum.algorithms.primitives.thc_chemistry_prepare import PrepareTHCChemistry
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost,
    estimate_component_resources,
)
from flagquantum.compiler.registers.lookup.source_qroam_clean import (
    optimal_qroam_clean_log_block_size,
)
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signatures(items: Sequence[tuple[str, Signature]]) -> None:
    for name, signature in items:
        display(SVG(render_call_graph_svg(export_signature_graph_dot(signature, name))))


## SELECT provider

A native directional emitter accepts named selection and target ports together
with explicit borrowed scratch and result slots. The selector preserves its
selection word and applies the indexed unitary to its target. Existing
SelectBlockEncoding owns this binding and its ordinary append consumer.


In [ ]:
# SelectBlockEncoding binds an existing directional SELECT provider.


## PrepareOracle

The existing preparation declaration owns selection, logical junk and coefficient
normalization. Its packed native provider binds explicit ports and independently
supplies both directions. Logical preparation junk can remain entangled with
selection, and clean scratch has its own lifetime.


In [ ]:
assert issubclass(IdentityPrepare, PrepareOracle)


## QubitizationWalk

The native walk composes SELECT with the shared prepared-state reflection.
It preserves named selection unions and explicit block junk. Its mathematical
inverse reverses the two stages. External controls reuse the shared predicate
binder and specialized positive-controlled SELECT.

The three examples retain the original Ising, THC and sparse-chemistry parameter
sets. Their graphs are declaration views. The THC example constructs complete
prepared data and a typed walk signature. Binding its physical SELECT requires
explicit rotation networks under the chemistry module's existing contract.
Source narrow-reflection and controlled-LCU signature failures remain recorded
in the module audits. The source slow four-site simulation remains unverified
after the bounded reference run. The final small executable example has a
complete complex-state check and one clean anchor.


In [ ]:
# QubitizationWalk composes accepted native block and reflection owners.


### Example Instances

In [ ]:
ising_example, hamiltonian = ising_walk(4, 0.2)
walk_op = ising_example.walk
alias_shape = AliasPreparationShape.from_table(ising_example.table)
walk_signature = source_walk_signature(
    alias_shape.selection_registers,
    alias_shape.selection_registers,
    (Register("target", opaque_register_type(4)),),
    (),
)


In [ ]:
# Reproduce the frozen RandomState(7) integral fixture in its original order.
num_spinorb = 152
num_bits_state_prep = 10
num_bits_rot = 20
thc_dim = 450
num_spat = num_spinorb // 2
rng = np.random.RandomState(7)
one_body = rng.normal(size=(num_spat, num_spat))
one_body = 0.5 * (one_body + one_body.T)
t_l = np.linalg.eigvalsh(one_body)
zeta = rng.normal(size=(thc_dim, thc_dim))
zeta = 0.5 * (zeta + zeta.T)
eta = rng.normal(size=(thc_dim, num_spat))
# The shared block minimizer has the same argmin as L/B+B at target width one.
qroam_blocking_factor = 1 << optimal_qroam_clean_log_block_size(thc_dim + num_spat, 1)
thc_data = thc_chemistry_data(t_l, eta, zeta, probability_bits=num_bits_state_prep)
thc_prepare = PrepareTHCChemistry(thc_data)
thc_ports = tuple(select_thc_signature(thc_dim, num_spinorb, num_bits_state_prep))
thc_signature = source_walk_signature(
    thc_ports[:-2], thc_prepare.selection_registers, thc_ports[-2:], ()
)


In [ ]:
num_spin_orb = 8
num_bits_rot_aa = 8
num_bits_state_prep = 12
spatial = num_spin_orb // 2
rng = np.random.RandomState(7)
tpq = rng.normal(size=(spatial, spatial))
tpq = 0.5 * (tpq + tpq.T)
eris = rng.normal(scale=4, size=(spatial,) * 4)
eris += np.transpose(eris, (0, 1, 3, 2))
eris += np.transpose(eris, (1, 0, 2, 3))
eris += np.transpose(eris, (2, 3, 0, 1))
walk_op_chem_sparse = SparseChemistryWalk.from_integrals(
    tpq, eris, rotation_bits=num_bits_rot_aa, probability_bits=num_bits_state_prep
)
sparse_signature = walk_op_chem_sparse.source_signature


#### Graphical Signature

In [ ]:
show_signatures(
    [
        ("Periodic Ising walk", walk_signature),
        ("THC walk declaration", thc_signature),
        ("Sparse chemistry walk", sparse_signature),
    ]
)


### Call Graph

In [ ]:
walk_calls = {"SELECT block": 1, "Prepared-state reflection": 1}
walk_graph = symbolic_call_graph(
    "Qubitization walk", {"Qubitization walk": walk_calls}, max_depth=1
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(walk_graph.root, walk_graph.direct_calls)
        )
    )
)
display(walk_graph.leaf_counts)
modeled_resources = estimate_component_resources(ising_example.resource_model())
display(modeled_resources)


## Executable append example
A two-row X/Z selector and an identity signal give a full three-qubit state with a clean borrowed anchor. Count emitted primitives independently from the source declaration model.


In [ ]:
pauli = SelectPauliLCU(1, 1, ("X", "Z"))


def select_provider(
    ports: Mapping[str, tuple[int, ...]],
    workspace: tuple[int, ...],
    classical_bits: tuple[int, ...],
    adjoint: bool,
) -> tuple[Instruction, ...]:
    return pauli.make_bounded_on(
        selection=ports["selection"],
        target=ports["target"],
        workspace=workspace,
        classical_bits=classical_bits,
    ).emit()


signal = IdentityPrepare((Register("selection", opaque_register_type(1)),)).bind()
small = QubitizationWalk(
    SelectBlockEncoding((("selection", 1),), (("target", 1),), select_provider, signal)
)
op = small.make_on(
    selection={"selection": (0,)}, target=(1,), reflection_workspace=(2,)
)
circuit = Circuit(3, dtype=torch.complex128).h(0).h(1)
circuit.append(op)
measured_primitives = dict(Counter(g.name for g in op.instructions))
selector_cost = pauli.make_bounded_on(
    selection=(0,), target=(1,), workspace=(), classical_bits=()
).resource_model()
identity_cost = ComponentCost("Identity signal", 1)
small_model = estimate_component_resources(
    small.resource_model(selector_cost, identity_cost, identity_cost)
)
display(measured_primitives)
display(small_model)
